In [ ]:
"""
random_sample_metadata.py

Randomly sample points from a CSV of lat/lng (e.g. greater_london_grid_50m_full.csv).
For each sampled point, call the Google Street View metadata API (source=outdoor, radius=radius).
If metadata.status == "OK", save the metadata JSON into Azure Blob Storage under:
  <container>/<prefix>/RandomSamplingMetadata/<sequential>_<point_idx>.json
Also save a local copy in ./RandomSamplingMetadata/.

Uses a checkpoint file 'random_sampling_checkpoint.json' to resume safely.

Requirements:
  pip install requests azure-storage-blob tqdm
"""
!pip install requests azure-storage-blob tqdm
import os
import csv
import json
import time
import random
import logging
import threading
import requests
from urllib.parse import urlencode
from typing import List, Tuple
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, wait, FIRST_COMPLETED # Added wait/FIRST_COMPLETED
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from azure.storage.blob import BlobServiceClient, ContentSettings
from azure.core.pipeline.transport import RequestsTransport
# -----------------------
# USER CONFIG (edit as needed)
# -----------------------
street_view_api_key = os.environ.get("STREET_VIEW_API_KEY", "")
azure_api_key = os.environ.get("AZURE_CONNECTION_STRING", "")
blob_path = ".../london/training"           # "<container>/<optional/prefix>"
locations_csv = "greater_london_grid_50m_full.csv"
radius = 50                                # meters (search radius for metadata)
source = "outdoor"
target_count = 56497                      # number of metadata hits to collect
request_timeout = 30
max_retries = 3
angles = [0, 90, 180, 270]  # headings (degrees) to check per point
pitch = 0
user_agent = "sv-random-sampler/1.0"

# Checkpoint + local metadata folder
checkpoint_local = "random_sampling_checkpoint.json"
local_meta_folder = "RandomSamplingMetadata56k"   # local folder for metadata jsons

# Azure blob folders (under prefix)
remote_meta_folder = "RandomSamplingMetadata56k"

max_workers = int(os.environ.get("RS_MAX_WORKERS", "12"))  # increase/decrease based on quota
max_requests_per_second = float(os.environ.get("RS_RPS", "0"))  # optional rate limit (per-second)
requests_pool_size = max(10, max_workers + 20)

# -----------------------
# logging
# -----------------------
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s: %(message)s")
logger = logging.getLogger("random_sampler")

# -----------------------
# Helpers: load CSV points
# -----------------------
def create_shared_session(pool_maxsize=requests_pool_size, max_retries_local=int(max_retries)):
    s = requests.Session()
    retry = Retry(
        total=max_retries_local,
        backoff_factor=0.8,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods={"GET"},
        raise_on_status=False
    )
    adapter = HTTPAdapter(max_retries=retry, pool_maxsize=pool_maxsize, pool_connections=pool_maxsize)
    s.mount("https://", adapter)
    s.mount("http://", adapter)
    s.headers.update({"User-Agent": user_agent})
    return s

# Create one shared requests.Session used both for StreetView API calls and Azure uploads
shared_session = create_shared_session(pool_maxsize=requests_pool_size, max_retries_local=int(max_retries))

def make_blob_service_client_with_transport(conn_str: str, shared_sess: requests.Session = None) -> BlobServiceClient:
    transport = RequestsTransport(session=shared_sess) if shared_sess is not None else None
    # Use from_connection_string explicitly (we expect a connection string)
    return BlobServiceClient.from_connection_string(conn_str, transport=transport)

# Use the shared session for your Google fetches too
session = shared_session

# Build BlobServiceClient with the shared session transport
blob_service = make_blob_service_client_with_transport(azure_api_key, shared_sess=shared_session)
container_name, *rest = blob_path.split("/", 1)
prefix = rest[0].rstrip("/") if rest else ""
container_client = blob_service.get_container_client(container_name)
try:
    container_client.create_container()
    logger.info("Created container: %s", container_name)
except Exception:
    logger.debug("Container may already exist (continuing).")

def load_locations_from_csv(path: str) -> List[Tuple[float, float]]:
    locs = []
    with open(path, newline="", encoding="utf-8") as fh:
        sample = fh.read(4096)
        fh.seek(0)
        sniffer = csv.Sniffer()
        has_header = False
        try:
            has_header = sniffer.has_header(sample)
        except Exception:
            has_header = False

        if has_header:
            fh.seek(0)
            dreader = csv.DictReader(fh)
            fieldmap = {name.lower(): name for name in dreader.fieldnames if name}
            lat_key = fieldmap.get("lat") or fieldmap.get("latitude") or None
            lon_key = fieldmap.get("lon") or fieldmap.get("lng") or fieldmap.get("longitude") or None
            if lat_key and lon_key:
                for row in dreader:
                    try:
                        lat = float(row[lat_key]); lon = float(row[lon_key])
                        locs.append((lat, lon))
                    except Exception:
                        continue
            else:
                fh.seek(0)
                reader2 = csv.reader(fh)
                next(reader2, None)
                for cols in reader2:
                    if len(cols) >= 2:
                        try:
                            lat = float(cols[0]); lon = float(cols[1])
                            locs.append((lat, lon))
                        except Exception:
                            continue
        else:
            fh.seek(0)
            reader = csv.reader(fh)
            for cols in reader:
                if len(cols) >= 2:
                    try:
                        lat = float(cols[0]); lon = float(cols[1])
                        locs.append((lat, lon))
                    except Exception:
                        continue
    return locs

# -----------------------
# Azure helpers
# -----------------------
def make_blob_service_client(azure_key: str) -> BlobServiceClient:
    if azure_key is None:
        raise ValueError("azure_api_key not provided")
    if azure_key.startswith("DefaultEndpointsProtocol=") or "AccountName=" in azure_key:
        return BlobServiceClient.from_connection_string(azure_key)
    if azure_key.startswith("https://") and "sv=" in azure_key:
        account_url = azure_key.split("?", 1)[0]
        sas_token = azure_key.split("?", 1)[1]
        return BlobServiceClient(account_url=account_url, credential=sas_token)
    if ":" in azure_key and len(azure_key.split(":", 1)[1]) > 8:
        account, key = azure_key.split(":", 1)
        conn_str = f"DefaultEndpointsProtocol=https;AccountName={account};AccountKey={key};EndpointSuffix=core.windows.net"
        return BlobServiceClient.from_connection_string(conn_str)
    raise ValueError("Unrecognized azure_api_key format. Provide connection string, SAS URL, or account:account_key")

# Create blob client once (same style as earlier)
# blob_service = make_blob_service_client(azure_api_key)
# container_name, *rest = blob_path.split("/", 1)
# prefix = rest[0].rstrip("/") if rest else ""
# container_client = blob_service.get_container_client(container_name)
try:
    container_client.create_container()
    logger.info("Created container: %s", container_name)
except Exception:
    logger.debug("Container may already exist (continuing).")

def upload_text_to_blob(text: str, blob_rel_path: str, content_type="application/json"):
    # quick wrapper (thread-safe per azure sdk)
    blob_client = container_client.get_blob_client(blob_rel_path)
    blob_client.upload_blob(text.encode("utf-8"), overwrite=True, content_settings=ContentSettings(content_type=content_type))

# -----------------------
# Google Street View metadata helpers (pooled session + retries)
# -----------------------
METADATA_BASE = "https://maps.googleapis.com/maps/api/streetview/metadata"

def build_metadata_url(lat: float, lng: float, radius: int = None, source: str = "outdoor"):
    r = int(radius) if radius is not None else int(globals().get("radius", 50))
    params = {"location": f"{lat},{lng}", "radius": r, "key": street_view_api_key, "source": source}
    return f"{METADATA_BASE}?{urlencode(params)}"

def create_session(max_retries_local=int(max_retries), pool_maxsize=100):
    s = requests.Session()
    # Configure urllib3 Retry for idempotent GETs
    retry = Retry(
        total=max_retries_local,
        backoff_factor=0.8,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods={"GET"},
        raise_on_status=False
    )
    adapter = HTTPAdapter(max_retries=retry, pool_maxsize=pool_maxsize, pool_connections=pool_maxsize)
    s.mount("https://", adapter)
    s.mount("http://", adapter)
    s.headers.update({"User-Agent": user_agent})
    return s

# Shared session for all workers
session = shared_session

def fetch_metadata(lat: float, lng: float, radius: int = None, retries: int = max_retries):
    """Uses the shared session. Raises exception on total failure"""
    url = build_metadata_url(lat, lng, radius=radius, source=source)
    last_exc = None
    for attempt in range(1, retries + 1):
        try:
            r = session.get(url, timeout=request_timeout)
            if r.status_code == 200:
                return r.json()
            last_exc = Exception(f"HTTP {r.status_code}: {r.text[:200]}")
        except Exception as e:
            last_exc = e
        # exponential sleep with jitter
        time.sleep(min(5, (1.2 ** attempt)) * (0.8 + random.random() * 0.4))
    raise last_exc


# -----------------------
# Checkpoint management (thread-safe)
# -----------------------
cp_lock = threading.Lock()

def load_checkpoint():
    if os.path.exists(checkpoint_local):
        try:
            with open(checkpoint_local, "r", encoding="utf-8") as fh:
                return json.load(fh)
        except Exception:
            logger.exception("Failed reading checkpoint; starting fresh.")
    return {"collected": {}, "count": 0}

cp_lock = threading.RLock()

def save_checkpoint(cp):
    """Writes locally under lock, then uploads to Azure WITHOUT holding the lock."""
    try:
        with cp_lock:
            with open(checkpoint_local, "w", encoding="utf-8") as fh:
                json.dump(cp, fh, indent=2)
    except Exception:
        logger.exception("Failed to save local checkpoint.")

    # UPLOAD HAPPENS HERE - OUTSIDE THE LOCK
    try:
        blob_rel = f"{prefix}/checkpoint/{checkpoint_local}" if prefix else f"checkpoint/{checkpoint_local}"
        upload_text_to_blob(json.dumps(cp), blob_rel)
    except Exception:
        logger.debug("Azure checkpoint sync delayed or failed.")

# -----------------------
# Optional simple rate limiter (token bucket)
# -----------------------
class TokenBucket:
    def __init__(self, rate_per_sec: float):
        self.rate = rate_per_sec
        self._tokens = rate_per_sec
        self._last = time.monotonic()
        self._lock = threading.Lock()

    def consume(self, amount: float = 1.0):
        if self.rate <= 0:
            return True
        with self._lock:
            now = time.monotonic()
            elapsed = now - self._last
            self._last = now
            self._tokens += elapsed * self.rate
            if self._tokens > self.rate:
                self._tokens = self.rate
            if self._tokens >= amount:
                self._tokens -= amount
                return True
            return False

    def wait_for_token(self):
        while not self.consume(1.0):
            time.sleep(0.01)

rate_limiter = TokenBucket(max_requests_per_second if max_requests_per_second and max_requests_per_second > 0 else 0)

# -----------------------
# Worker: process a single point
# -----------------------
def process_point(point_idx: int, lat: float, lng: float, cp: dict, pbar: tqdm) -> bool:
    key = str(point_idx)

    # Quick check if done
    with cp_lock:
        if key in cp.get("collected", {}):
            return False

    rate_limiter.wait_for_token()

    try:
        md = fetch_metadata(lat, lng, radius=radius)
    except Exception as e:
        return False

    if md.get("status") != "OK":
        return False

    # Metadata record
    rec = {
        "point_idx": point_idx,
        "lat": lat,
        "lng": lng,
        "metadata_response": md,
        "saved_at": time.time()
    }

    # 1. Local Disk Write (Fast)
    local_name = os.path.join(local_meta_folder, f"{point_idx}.json")
    try:
        with open(local_name, "w", encoding="utf-8") as fh:
            json.dump(rec, fh, indent=2)
    except Exception:
        return False

    # 2. Azure Metadata Upload (Network I/O)
    # We do this BEFORE updating the checkpoint so we don't mark as 'done' if upload fails
    blob_rel = f"{prefix}/{remote_meta_folder}/{point_idx}.json" if prefix else f"{remote_meta_folder}/{point_idx}.json"
    try:
        upload_text_to_blob(json.dumps(rec, indent=2), blob_rel)
    except Exception as e:
        logger.error(f"Upload failed for {point_idx}: {e}")
        return False

    # 3. Update Checkpoint (Lock only for dict update and local file)
    with cp_lock:
        if key not in cp["collected"]:
            cp["collected"][key] = blob_rel
            cp["count"] = cp.get("count", 0) + 1
            # save_checkpoint handles its own lock internally
            save_checkpoint(cp)
            pbar.update(1)

    return True

# -----------------------
# Main sampling routine (concurrent)
# -----------------------
def main():
    os.makedirs(local_meta_folder, exist_ok=True)
    locs = load_locations_from_csv(locations_csv)
    total_points = len(locs)
    if total_points == 0:
        raise SystemExit("No locations loaded from CSV.")

    logger.info("Loaded %d points from CSV.", total_points)
    cp = load_checkpoint()
    collected_count = int(cp.get("count", 0))
    collected_set = set(cp.get("collected", {}).keys())

    logger.info("Already collected %d metadata entries according to checkpoint.", collected_count)

    all_indices = list(range(total_points))
    remaining_indices = [i for i in all_indices if str(i) not in collected_set]
    random.shuffle(remaining_indices)

    # Progress bar
    pbar = tqdm(total=target_count, initial=collected_count, desc="metadata collected", unit="meta")

    # If target already satisfied
    if collected_count >= target_count:
        pbar.close()
        logger.info("Target already reached: %d >= %d", collected_count, target_count)
        return

    # We'll use ThreadPoolExecutor to concurrently process points.
    # Submit larger than target_count to keep pipeline full; we stop when cp["count"] reaches target_count.
    max_workers_local = max(2, min(max_workers, len(remaining_indices)))
    logger.info("Starting with max_workers=%d", max_workers_local)

    idx_iter_lock = threading.Lock()
    idx_pos = {"pos": 0}  # mutable closure for current index pointer

    def get_next_index():
        with idx_iter_lock:
            if idx_pos["pos"] >= len(remaining_indices):
                return None
            v = remaining_indices[idx_pos["pos"]]
            idx_pos["pos"] += 1
            return v

    # -----------------------
    # Thread pool execution
    # -----------------------
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_idx = {}

        # Initial fill: max_workers * 2 keeps the queue healthy
        for _ in range(min(len(remaining_indices), max_workers * 2)):
            idx = get_next_index()
            if idx is None: break
            lat, lng = locs[idx]
            future_to_idx[executor.submit(process_point, idx, lat, lng, cp, pbar)] = idx

        while future_to_idx:
            # Wait for at least one worker to finish
            done, _ = wait(future_to_idx.keys(), return_when=FIRST_COMPLETED)

            for f in done:
                future_to_idx.pop(f)
                try:
                    f.result()
                except Exception as e:
                    logger.error(f"Worker crashed: {e}")

                # Stop condition
                with cp_lock:
                    if cp.get("count", 0) >= target_count:
                        logger.info("Target reached. Cleaning up...")
                        # Cancel remaining futures
                        for pending_f in future_to_idx.keys():
                            pending_f.cancel()
                        return

                # Replenish with a new point
                next_idx = get_next_index()
                if next_idx is not None:
                    lat, lng = locs[next_idx]
                    future_to_idx[executor.submit(process_point, next_idx, lat, lng, cp, pbar)] = next_idx

    pbar.close()
    logger.info("Finished sampling. collected=%d", cp.get("count", 0))

if __name__ == "__main__":
    main()


In [ ]:
from google.colab import drive
drive.mount('/content/drive')